In [ ]:
# author: Luka Kowalchuk
# student nr: 13326015
# year: 2025-2026

# This notebook was used to train and evaluate a Vision Transformer (ViT) to rapidly classify forensically relevant insects.
# The ViT forms a part of the Research Project:
# A Comparative Evaluation of Vision Transformer and Convolutional Neural Network Performance for Forensically Relevant Dipterans Classification
# as a part of the MSc Forensic Science at the University of Amsterdam, collaborating with the Wildlife Forensic Academy in South Africa
# The model is pretrained on BioCLIP: Samuel Stevens et al. “BioCLIP: A Vision Foundation Model for the Tree of Life”. In: Proceedings
# of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR). June 2024, pp. 19412–19424.
# Low Rank Adaptation Parameter Efficient Finetuning is applied using a dataset developed by the Wildlife Forensic Academy
# to represent forensically relevant dipterans species (adults and maggots) in South Africa
# The model was ran in Google Colab, hence the cells are designed for this purpose, but it can be ran locally with minor changes
# Claude Sonnet 4.6 was used to create this notebook, which was monitored and modified consistenly using unit-checks and other tests

# Make sure to check Cell 4 before running everything

In [ ]:
# Cell 1 — Install dependencies

# NOTE: torch, torchvision, Pillow, and numpy are pre-installed in Google Colab.
# If running locally, install PyTorch manually first for GPU support:
# https://pytorch.org/get-started/locally/
!pip install open_clip_torch peft scikit-learn matplotlib tqdm umap-learn plotly torchvision Pillow numpy -q
!pip install -q --upgrade torchao

In [ ]:
# Cell 2 — Mount Google Drive & unzip dataset

from google.colab import drive
drive.mount('/content/drive')

# Unzipping the final dataset or any dataset in the local directory seemed to improve speed during the preprossessing and training
# !unzip /content/drive/MyDrive/final_dataset.zip -d /content/ # Change the directory if another dataset should be featured

In [ ]:
# Cell 3 — Imports

import os
import shutil
import random
import json
from collections import defaultdict, Counter
from io import BytesIO
import base64

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.transforms import functional as TF


import open_clip
from peft import LoraConfig, get_peft_model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
import umap
import plotly.graph_objects as go
from IPython.display import display, HTML
from tqdm import tqdm

In [ ]:
# Cell 4 — CONFIG
# ============================================================
# This is the only cell you need to edit between runs.
# ============================================================

# --- MODE ---
# Choose 'flies' or 'maggots' to select which subset to train on.
MODE = 'flies'

# --- PATHS ---
DATASET_PATH = '/content/final_dataset'                                        # path to unzipped dataset
AUG_DIR      = f'/content/drive/MyDrive/vit_16_augmented_dataset_{MODE}'             # where augmented images are saved
CHECKPOINT   = f'/content/drive/MyDrive/vit_16_{MODE}.pt'          # where best model is saved
PLOT_DIR     = f'/content/drive/MyDrive/vit_16_plots_{MODE}'                         # where plots are saved
os.makedirs(PLOT_DIR, exist_ok=True)

# --- AUGMENTATIONS ---
# Add or remove augmentations as needed. Each entry is a function that takes a PIL image and returns a PIL image.
AUGMENTATIONS = [
    # lambda img: TF.rotate(img, 90),
    # lambda img: TF.hflip(TF.rotate(img, 180)),
]

# --- TRAINING HYPERPARAMETERS ---
BATCH_SIZE        = 16
EPOCHS            = 10
LEARNING_RATE     = 1e-4
WEIGHT_DECAY      = 0.01
TRIPLET_MARGIN    = 0.2

# --- LORA HYPERPARAMETERS ---
LORA_RANK         = 8
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.1

# --- DATASET PARAMETERS ---
MIN_IMAGES             = 282    # minimum original images required for a class to be included
TRIAL_IMAGES_PER_CLASS = 282    # number of originals sampled per class for the trial dataset

In [ ]:
# Cell 5 — SafeImageFolder

class SafeImageFolder(ImageFolder):
    """
    Extends torchvision's ImageFolder to filter out corrupt images on load.
    Set verify=False to skip the scan (e.g. for already-validated datasets).
    """
    def __init__(self, root, transform=None, verify=True):
        super().__init__(root, transform=transform)

        if verify:
            original_count = len(self.samples)
            valid_samples  = []

            for path, label in self.samples:
                try:
                    img = Image.open(path)
                    img.verify()
                    valid_samples.append((path, label))
                except (UnidentifiedImageError, OSError, Exception):
                    print(f"  Removing corrupt image: {path}")

            self.samples = valid_samples
            self.imgs    = valid_samples
            removed      = original_count - len(valid_samples)
            print(f"  Dataset ready: {len(valid_samples)} valid images ({removed} corrupt removed)")

    def __getitem__(self, index):
        path, label  = self.samples[index]
        sample       = self.loader(path)
        if self.transform is not None:
            sample = self.transform(sample)
        return sample, label

In [ ]:
# =============================================================================
# 6. LOAD ViT-B/16 BACKBONE (ImageNet-1k)
# =============================================================================
import torchvision.models as models
import torchvision.transforms as transforms

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

# 1. Define Preprocessing (Standard ImageNet-1k requirements for ViT)
preprocess = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 2. Load the ViT-B/16 Model
# We use IMAGENET1K_V1 to ensure a fair comparison with EfficientNet
_base_model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1).to(device)
_base_model.eval()

# 3. Create a Wrapper to mimic the BioCLIP/OpenCLIP API
class ViTWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        # Remove the classification head to get the 768-dim features
        self.model.heads = nn.Identity()

    def encode_image(self, x):
        # Torchvision ViT returns the CLS token embedding when heads is Identity
        return self.model(x)

    def forward(self, x):
        return self.encode_image(x)

model = ViTWrapper(_base_model)
print("ViT-B/16 (ImageNet-1k) loaded and wrapped for BioCLIP compatibility.")

In [ ]:
# =============================================================================
# CELL 7 — LOAD DATASET
# =============================================================================

dataset     = SafeImageFolder(DATASET_PATH, transform=preprocess)
class_names = dataset.classes
num_classes = len(class_names)

In [ ]:
# =============================================================================
# Cell 8 — DATA AUGMENTATION
# =============================================================================

# Generates augmented versions per image based on AUGMENTATIONS defined in CONFIG.
# Only processes fly or maggot classes depending on MODE.
# Output is saved to AUG_DIR and only needs to be run once per MODE.

# Filter classes by MODE
eligible_classes = defaultdict(list)
for path, label in dataset.samples:
    class_name = dataset.classes[label].lower()
    if MODE == 'maggots' and 'maggot' in class_name:
        eligible_classes[label].append((path, label))
    elif MODE == 'flies' and 'maggot' not in class_name:
        eligible_classes[label].append((path, label))

# Save originals + augmentations to AUG_DIR
total_saved = 0
for label, samples in sorted(eligible_classes.items()):
    class_name = dataset.classes[label]
    out_dir    = os.path.join(AUG_DIR, class_name)
    os.makedirs(out_dir, exist_ok=True)

    for path, _ in tqdm(samples, desc=class_name):
        img = Image.open(path).convert('RGB')
        img = img.resize((512, 512), Image.LANCZOS)
        img.save(os.path.join(out_dir, os.path.basename(path)))
        total_saved += 1

        for i, aug_fn in enumerate(AUGMENTATIONS):
            aug_img = aug_fn(img)
            fname   = f"{os.path.splitext(os.path.basename(path))[0]}_aug{i+1}.jpg"
            aug_img.save(os.path.join(out_dir, fname))
            total_saved += 1

print(f"Done. {total_saved} images saved to: {AUG_DIR}")

In [ ]:
# =============================================================================
# Cell 8b. DEFINE UNSEEN FOLDER
# =============================================================================

# 1. Configuration
# Assuming AUG_DIR is already defined as "/content/drive/MyDrive/vit_16_augmente"
# Define your UNSEEN_DIR alongside it
UNSEEN_DIR = os.path.join(os.path.dirname(AUG_DIR), "unseen_vit_16")

# Make sure the unseen directory exists
os.makedirs(UNSEEN_DIR, exist_ok=True)

moved_classes = []
kept_classes = []

print(f"Scanning classes in: {AUG_DIR}...\n")

# 2. Iterate through each class folder in the augmented directory
for class_name in sorted(os.listdir(AUG_DIR)):
    class_path = os.path.join(AUG_DIR, class_name)

    # Ensure it's a directory
    if os.path.isdir(class_path):
        num_images = len(os.listdir(class_path))

        # 3. Check threshold and route
        if num_images < MIN_IMAGES:
            # Move the entire folder to UNSEEN_DIR
            destination_path = os.path.join(UNSEEN_DIR, class_name)
            shutil.move(class_path, destination_path)
            moved_classes.append((class_name, num_images))
            print(f" 📦 MOVED: {class_name} ({num_images} images) -> unseen")
        else:
            kept_classes.append((class_name, num_images))
            print(f" ✅ KEPT:  {class_name} ({num_images} images) -> augmented")

# 4. Summary
print(f"\n--- SUMMARY (Threshold: {MIN_IMAGES}) ---")
print(f"Classes kept for training: {len(kept_classes)}")
print(f"Classes moved to unseen:   {len(moved_classes)}")
print(f"Unseen directory: {UNSEEN_DIR}")

In [ ]:
# =============================================================================
# CELL 9 — BUILD TRIAL DATASET
# =============================================================================
# Loads the augmented dataset and builds a balanced trial dataset by:
# 1. Filtering classes by MODE and MIN_IMAGES
# 2. Sampling TRIAL_IMAGES_PER_CLASS originals per class
# 3. Including all augmented versions of the sampled originals

# Load augmented dataset
aug_dataset = SafeImageFolder(AUG_DIR, transform=preprocess, verify=False)
print(f"Augmented dataset: {len(aug_dataset)} images across {len(aug_dataset.classes)} classes")

# Separate originals and augmentations by filename
originals_by_class     = defaultdict(list)
augmentations_by_class = defaultdict(list)

for path, label in aug_dataset.samples:
    if '_aug' in os.path.basename(path):
        augmentations_by_class[label].append((path, label))
    else:
        originals_by_class[label].append((path, label))

# Filter classes by MODE and MIN_IMAGES
eligible_classes = {
    label: samples
    for label, samples in originals_by_class.items()
    if len(samples) >= MIN_IMAGES
    and (
        ('maggot' in aug_dataset.classes[label].lower() and MODE == 'maggots')
        or
        ('maggot' not in aug_dataset.classes[label].lower() and MODE == 'flies')
    )
}

print(f"\nClasses included: {len(eligible_classes)}")
for label in sorted(eligible_classes.keys()):
    print(f"  [{label:2d}] {aug_dataset.classes[label]:45s} originals: {len(originals_by_class[label])}  augmented: {len(augmentations_by_class[label])}")

# Sample TRIAL_IMAGES_PER_CLASS originals per class
# and include all their corresponding augmentations
random.seed(42)
trial_samples = []

for label in sorted(eligible_classes.keys()):
    sampled_originals = random.sample(originals_by_class[label], TRIAL_IMAGES_PER_CLASS)
    trial_samples.extend(sampled_originals)

    sampled_stems = {os.path.splitext(os.path.basename(p))[0] for p, _ in sampled_originals}
    for path, lbl in augmentations_by_class[label]:
        stem = os.path.splitext(os.path.basename(path))[0]
        for i in range(1, len(AUGMENTATIONS) + 1):
            stem = stem.replace(f'_aug{i}', '')
        if stem in sampled_stems:
            trial_samples.append((path, lbl))

print(f"\nTrial dataset: {len(trial_samples)} images across {len(eligible_classes)} classes")
print(f"Expected: {TRIAL_IMAGES_PER_CLASS * (len(AUGMENTATIONS) + 1) * len(eligible_classes)} ({TRIAL_IMAGES_PER_CLASS} × {len(AUGMENTATIONS) + 1} × {len(eligible_classes)} classes)")

In [ ]:
# =============================================================================
# CELL 10 — TRAIN/VAL/TEST-SPLIT
# =============================================================================

# Splits the trial dataset 80/10/10 into train, val, and test sets.
# Originals and their augmentations are always kept in the same split
# to prevent data leakage.

random.seed(42)

# Group each original and its augmentations together by filename stem
groups = defaultdict(list)
for path, label in trial_samples:
    fname = os.path.basename(path)
    stem  = fname
    for i in range(1, len(AUGMENTATIONS) + 1):
        stem = stem.replace(f'_aug{i}', '')
    stem = os.path.splitext(stem)[0]
    groups[(label, stem)].append((path, label))

# Split groups 80/10/10 per class
train_samples, val_samples, test_samples = [], [], []

by_class = defaultdict(list)
for key, samples in groups.items():
    label = key[0]
    by_class[label].append(samples)

for label, sample_groups in sorted(by_class.items()):
    random.shuffle(sample_groups)
    n         = len(sample_groups)
    train_end = int(0.8 * n)
    val_end   = int(0.9 * n)

    for group in sample_groups[:train_end]:
        train_samples.extend(group)
    for group in sample_groups[train_end:val_end]:
        val_samples.extend(group)
    for group in sample_groups[val_end:]:
        test_samples.extend(group)

print(f"Train: {len(train_samples)} images")
print(f"Val:   {len(val_samples)} images")
print(f"Test:  {len(test_samples)} images")

In [ ]:
# =============================================================================
# CELL 11 — EMBEDDING EXTRACTION UTILITY
# =============================================================================

# Extracts L2-normalised image embeddings from the model for a given set of samples.
# Used for both baseline evaluation and post-training evaluation.

class EvalDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img         = self.transform(Image.open(path).convert('RGB'))
        return img, label

def extract_embeddings(samples, transform, model, device):
    dataset = EvalDataset(samples, transform)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    all_embeddings = []
    all_labels     = []

    model.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Extracting embeddings"):
            imgs = imgs.to(device)
            emb  = model.encode_image(imgs)
            emb  = emb / emb.norm(dim=-1, keepdim=True)
            all_embeddings.append(emb.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.concatenate(all_embeddings, axis=0), np.concatenate(all_labels, axis=0)

print("extract_embeddings defined.")

In [ ]:
# =============================================================================
# BASELINE TEST (skip if the baseline evaluation is not necessary)
# =============================================================================

# Extract embeddings (no LoRA, base ViT)
print("Extracting train embeddings...")
train_embeddings, train_labels = extract_embeddings(train_samples, preprocess, model, device)

print("Extracting test embeddings...")
test_embeddings, test_labels = extract_embeddings(test_samples, preprocess, model, device)

# KNN classification (val as train, test as test)
print("\nRunning KNN classifier...")
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(train_embeddings, train_labels)
test_preds = knn.predict(test_embeddings)

# Accuracy
overall_acc = np.mean(test_preds == test_labels)
print(f"\nBaseline Accuracy (untrained ViT16): {overall_acc*100:.1f}%")

# Classification report
target_names = [aug_dataset.classes[i] for i in sorted(np.unique(test_labels))]
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=target_names))

# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title("Confusion Matrix — Baseline ViT16 (untrained)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"baseline_vit16_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

# t-SNE
print("\nComputing t-SNE...")
all_embeddings_combined = np.concatenate([train_embeddings, test_embeddings], axis=0)
all_labels_combined     = np.concatenate([train_labels, test_labels], axis=0)
all_splits              = np.array(['val'] * len(train_embeddings) + ['test'] * len(test_embeddings))

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings_combined)

fig, ax = plt.subplots(figsize=(14, 10))
colors = plt.colormaps['tab20'].resampled(len(np.unique(all_labels_combined)))
unique_labels = sorted(np.unique(all_labels_combined))

for i, label in enumerate(unique_labels):
    for split, marker, size in [('val', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels_combined == label) & (all_splits == split)
        ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   color=colors(i), marker=marker, s=size, alpha=0.6,
                   label=aug_dataset.classes[label] if split == 'val' else None)

ax.set_title("t-SNE — Baseline ViT16 (untrained)\n(circles = val, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"baseline_tsne_vit16_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

print("Done. Plots saved to Drive.")



In [ ]:
# =============================================================================
# CELL 12 — TRIPLET DATASET & DATALOADER
# =============================================================================

# For each anchor image, randomly samples a positive (same class)
# and a negative (different class) to form a triplet for training.

class TripletDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform

        self.by_class = defaultdict(list)
        for path, label in samples:
            self.by_class[label].append(path)
        self.labels = list(self.by_class.keys())

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        anchor_path, anchor_label = self.samples[idx]

        # Sample positive: same class, different image
        positive_path = anchor_path
        while positive_path == anchor_path:
            positive_path = random.choice(self.by_class[anchor_label])

        # Sample negative: different class
        negative_label = anchor_label
        while negative_label == anchor_label:
            negative_label = random.choice(self.labels)
        negative_path = random.choice(self.by_class[negative_label])

        anchor   = self.transform(Image.open(anchor_path).convert('RGB'))
        positive = self.transform(Image.open(positive_path).convert('RGB'))
        negative = self.transform(Image.open(negative_path).convert('RGB'))

        return anchor, positive, negative, anchor_label

train_dataset = TripletDataset(train_samples, transform=preprocess)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)

print(f"Triplet train dataset: {len(train_dataset)} triplets")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# =============================================================================
# CELL 13 - LORA SETUP (Tailored for Torchvision ViT-B/16)
# =============================================================================

# 1. Freeze all base model parameters
for param in model.parameters():
    param.requires_grad = False

# 2. Apply LoRA to the correct Torchvision layers
# - "out_proj" targets the output of the MultiheadAttention blocks
# - "mlp.0" and "mlp.3" target the two linear layers inside the Transformer MLP blocks
lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_ALPHA,
    target_modules = ["out_proj", "mlp.0", "mlp.3"],
    lora_dropout   = LORA_DROPOUT,
    bias           = "none",
)

model = get_peft_model(model, lora_config)

# 3. Sanity Check
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")

In [ ]:
# =============================================================================
# CELL 14 — TRIPLET LOSS WITH SEMI-HARD MINING
# =============================================================================

# For each triplet (anchor, positive, negative) there are three possible cases:
#   Easy:      d(a,n) > d(a,p) + margin  → loss = 0, model already separates them well
#   Semi-hard: d(a,n) > d(a,p) but d(a,n) < d(a,p) + margin  → best learning signal
#   Hard:      d(a,n) < d(a,p)  → negative is closer than positive, unstable early in training
# Only semi-hard triplets are back-propagated. Falls back to all non-easy if none are found.

class TripletLossWithMining(torch.nn.Module):
    def __init__(self, margin=TRIPLET_MARGIN):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        d_pos = torch.sum((anchor - positive) ** 2, dim=1)
        d_neg = torch.sum((anchor - negative) ** 2, dim=1)

        easy      = (d_neg > d_pos + self.margin)
        hard      = (d_neg < d_pos)
        semi_hard = (~easy & ~hard)

        losses = torch.clamp(d_pos - d_neg + self.margin, min=0.0)

        if semi_hard.sum() > 0:
            loss = losses[semi_hard].mean()
        else:
            non_easy = ~easy
            loss = losses[non_easy].mean() if non_easy.sum() > 0 else losses.mean()

        batch_size = anchor.shape[0]
        stats = {
            "easy":          easy.sum().item(),
            "semi_hard":     semi_hard.sum().item(),
            "hard":          hard.sum().item(),
            "easy_pct":      100 * easy.sum().item() / batch_size,
            "semi_hard_pct": 100 * semi_hard.sum().item() / batch_size,
            "hard_pct":      100 * hard.sum().item() / batch_size,
        }

        return loss, stats

triplet_loss_fn = TripletLossWithMining()
print(f"Triplet loss initialised with margin={TRIPLET_MARGIN}")

In [ ]:
# =============================================================================
# CELL 15 — TRAINING LOOP
# =============================================================================

# Trains the LoRA-adapted BioCLIP model using triplet loss.
# Saves the best model checkpoint to Drive based on validation accuracy.
# One could consider picking the best model based on lowest validation loss.
# Training history is saved to Drive at the end.

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr           = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY
)

best_val_acc = 0.0
history = {
    "train_loss":    [],
    "val_loss":      [],
    "train_acc":     [],
    "val_acc":       [],
    "easy_pct":      [],
    "semi_hard_pct": [],
    "hard_pct":      [],
}

def compute_val_loss(samples, model, device):
    """Computes average triplet loss on validation samples."""
    val_dataset = TripletDataset(samples, transform=preprocess)
    val_loader  = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            anchor, positive, negative, _ = batch
            anchor   = anchor.to(device)
            positive = positive.to(device)
            negative = negative.to(device)

            all_imgs = torch.cat([anchor, positive, negative], dim=0)
            all_embs = model.encode_image(all_imgs)
            anchor_emb, positive_emb, negative_emb = all_embs.chunk(3, dim=0)

            anchor_emb   = anchor_emb   / anchor_emb.norm(dim=-1, keepdim=True)
            positive_emb = positive_emb / positive_emb.norm(dim=-1, keepdim=True)
            negative_emb = negative_emb / negative_emb.norm(dim=-1, keepdim=True)

            loss, _ = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)
            total_loss += loss.item()

    return total_loss / len(val_loader)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss  = 0.0
    epoch_stats = {"easy": 0, "semi_hard": 0, "hard": 0}

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        anchor, positive, negative, _ = batch
        anchor   = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        all_imgs = torch.cat([anchor, positive, negative], dim=0)
        all_embs = model.encode_image(all_imgs)
        anchor_emb, positive_emb, negative_emb = all_embs.chunk(3, dim=0)

        anchor_emb   = anchor_emb   / anchor_emb.norm(dim=-1, keepdim=True)
        positive_emb = positive_emb / positive_emb.norm(dim=-1, keepdim=True)
        negative_emb = negative_emb / negative_emb.norm(dim=-1, keepdim=True)

        loss, stats = triplet_loss_fn(anchor_emb, positive_emb, negative_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        for k in epoch_stats:
            epoch_stats[k] += stats[k]

    # Compute metrics
    avg_train_loss = epoch_loss / len(train_loader)
    avg_val_loss   = compute_val_loss(val_samples, model, device)
    total_triplets = sum(epoch_stats.values())

    # Extract embeddings for KNN accuracy
    train_emb, train_lbl = extract_embeddings(train_samples, preprocess, model, device)
    val_emb,   val_lbl   = extract_embeddings(val_samples,   preprocess, model, device)

    # KNN accuracy
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn.fit(train_emb, train_lbl)
    train_acc = np.mean(knn.predict(train_emb) == train_lbl)
    val_acc   = np.mean(knn.predict(val_emb)   == val_lbl)

    # Update history
    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(avg_val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["easy_pct"].append(100 * epoch_stats["easy"] / total_triplets)
    history["semi_hard_pct"].append(100 * epoch_stats["semi_hard"] / total_triplets)
    history["hard_pct"].append(100 * epoch_stats["hard"] / total_triplets)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"  Train Acc:  {train_acc*100:.1f}%  | Val Acc:  {val_acc*100:.1f}%")
    print(f"  Easy: {history['easy_pct'][-1]:.1f}%  Semi-hard: {history['semi_hard_pct'][-1]:.1f}%  Hard: {history['hard_pct'][-1]:.1f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT)
        print(f"  Saved best model (val acc: {best_val_acc*100:.1f}%)")

# Load best model and extract final embeddings
print("\nLoading best model and extracting final embeddings...")
model.load_state_dict(torch.load(CHECKPOINT, map_location=device))
model.eval()

train_embeddings, train_labels = extract_embeddings(train_samples, preprocess, model, device)
val_embeddings,   val_labels   = extract_embeddings(val_samples,   preprocess, model, device)
test_embeddings,  test_labels  = extract_embeddings(test_samples,  preprocess, model, device)

# Save training history
with open(os.path.join(PLOT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f)

print(f"\nTraining complete. Best val acc: {best_val_acc*100:.1f}%")
print(f"Train embeddings: {train_embeddings.shape}")
print(f"Val embeddings:   {val_embeddings.shape}")
print(f"Test embeddings:  {test_embeddings.shape}")

In [ ]:
# =============================================================================
# CELL 16 — KNN CLASSIFICIATION & REPORT
# =============================================================================

# Classifies test embeddings using KNN trained on train embeddings.
# Prints overall accuracy and a full per-class classification report.

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(train_embeddings, train_labels)
test_preds  = knn.predict(test_embeddings)
overall_acc = np.mean(test_preds == test_labels)

print(f"Overall accuracy: {overall_acc*100:.1f}%")

target_names = [aug_dataset.classes[i] for i in sorted(np.unique(test_labels))]
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=target_names))

In [ ]:
# =============================================================================
# CELL 17 — CONFUSION MATRIX
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 12))
cm      = confusion_matrix(test_labels, test_preds)
disp    = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title(f"Confusion Matrix — ViT16 + LoRA ({MODE})", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"confusion_matrix_vit16_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 18 — T-SNE PLOT
# =============================================================================

# Visualizes train and test embeddings in 2D using t-SNE.
# Circles = train, stars = test.

all_embeddings = np.concatenate([train_embeddings, test_embeddings], axis=0)
all_labels     = np.concatenate([train_labels,     test_labels],     axis=0)
all_splits     = np.array(['train'] * len(train_embeddings) + ['test'] * len(test_embeddings))

print("Computing t-SNE...")
tsne          = TSNE(n_components=2, perplexity=50, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings)

unique_labels = sorted(np.unique(all_labels))
colors        = plt.colormaps['tab20'].resampled(len(unique_labels))

fig, ax = plt.subplots(figsize=(14, 10))
for i, label in enumerate(unique_labels):
    for split, marker, size in [('train', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels == label) & (all_splits == split)
        ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   color=colors(i), marker=marker, s=size, alpha=0.6,
                   label=aug_dataset.classes[label] if split == 'train' else None)

ax.set_title(f"t-SNE — ViT16 + LoRA ({MODE})\n(circles = train, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
# plt.savefig(os.path.join(PLOT_DIR, f"tsne_plot_vit16_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 19 — UMAP PLOT
# =============================================================================

# Visualizes train and test embeddings in 2D using UMAP.
# Circles = train, stars = test.

print("Computing UMAP...")
reducer       = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1, metric='cosine')
embeddings_2d_umap = reducer.fit_transform(all_embeddings)

fig, ax = plt.subplots(figsize=(14, 10))
for i, label in enumerate(unique_labels):
    for split, marker, size in [('train', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels == label) & (all_splits == split)
        ax.scatter(embeddings_2d_umap[mask, 0], embeddings_2d_umap[mask, 1],
                   color=colors(i), marker=marker, s=size, alpha=0.6,
                   label=aug_dataset.classes[label] if split == 'train' else None)

ax.set_title(f"UMAP — BioCLIP + LoRA ({MODE})\n(circles = train, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"umap_plot_vit16_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# Interactive Plotly t-SNE
# Interactive version of the t-SNE plot with hover details per point.
# Saved as a standalone HTML file to Drive.

label_to_name = {label: aug_dataset.classes[label] for label in unique_labels}
train_paths   = [path for path, _ in train_samples]
test_paths    = [path for path, _ in test_samples]
all_paths     = train_paths + test_paths

# Build hover text
hover_texts = []
for path, label, split in zip(all_paths, all_labels, all_splits):
    fname = os.path.basename(path)
    hover_texts.append(
        f"<b>{label_to_name[label]}</b><br>"
        f"Split: {split}<br>"
        f"File: {fname}<br>"
        f"{'[AUGMENTED]' if '_aug' in fname else '[ORIGINAL]'}"
    )

colors_hex = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']

fig = go.Figure()
for i, label in enumerate(unique_labels):
    color = colors_hex[i % len(colors_hex)]
    name  = label_to_name[label]
    for split, marker_symbol, marker_size in [('train', 'circle', 8), ('test', 'star', 12)]:
        mask    = (all_labels == label) & (all_splits == split)
        indices = np.where(mask)[0]
        fig.add_trace(go.Scatter(
            x            = embeddings_2d[mask, 0],
            y            = embeddings_2d[mask, 1],
            mode         = 'markers',
            marker       = dict(
                symbol  = marker_symbol,
                size    = marker_size,
                color   = color,
                opacity = 0.7 if split == 'train' else 0.95,
                line    = dict(width=0.5, color='black') if split == 'test' else dict(width=0),
            ),
            name         = f"{name} ({split})" if split == 'test' else name,
            legendgroup  = name,
            showlegend   = (split == 'train'),
            text         = [hover_texts[j] for j in indices],
            hovertemplate= "%{text}<extra></extra>",
        ))

fig.update_layout(
    title    = f"t-SNE — ViT16 + LoRA ({MODE})",
    width    = 950,
    height   = 700,
    legend   = dict(x=1.01, y=1, font=dict(size=10)),
    hovermode= 'closest',
    margin   = dict(r=200),
)

fig.write_html(os.path.join(PLOT_DIR, "tsne_interactive.html"))
fig.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 20 — TRAINING HISTORY PLOT
# =============================================================================

# Plots loss, accuracy and triplet mining statistics across epochs.

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Loss ---
axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker='o')
axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   marker='o')
axes[0].set_title("Loss per Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Triplet Loss")
axes[0].legend()
axes[0].grid(True)

# --- Accuracy ---
axes[1].plot(epochs, [a * 100 for a in history["train_acc"]], label="Train Acc", marker='o')
axes[1].plot(epochs, [a * 100 for a in history["val_acc"]],   label="Val Acc",   marker='o')
axes[1].set_title("Accuracy per Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True)

# --- Mining statistics ---
axes[2].plot(epochs, history["easy_pct"],      label="Easy",      marker='o')
axes[2].plot(epochs, history["semi_hard_pct"], label="Semi-Hard", marker='o')
axes[2].plot(epochs, history["hard_pct"],      label="Hard",      marker='o')
axes[2].set_title("Triplet Mining Statistics per Epoch")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Percentage (%)")
axes[2].legend()
axes[2].grid(True)

plt.suptitle(f"Training History — ViT16 + LoRA ({MODE})", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, "training_history_vit16.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# CELL 21 - FINAL EVALUATION: SEEN vs. UNSEEN (OOD DETECTION)
# =============================================================================

# 1. Define Unseen Samples (Loading from the UNSEEN_DIR)
unseen_samples = []
for root, dirs, files in os.walk(UNSEEN_DIR):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            path = os.path.join(root, f)
            # Use a dummy label (0) since we only care about 'unseen' status
            unseen_samples.append((path, 0))

if len(unseen_samples) == 0:
    print(f"⚠️ WARNING: No images found in {UNSEEN_DIR}. Did you run the post-process cell?")

def extract_embeddings_final(samples, desc=""):
    """Helper to extract L2-normalized embeddings from the current model."""
    model.eval()
    embeddings, labels = [], []
    for path, label in tqdm(samples, desc=desc):
        try:
            img = Image.open(path).convert("RGB")
            img_t = preprocess(img).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = model.encode_image(img_t)
                # L2 normalize for cosine similarity/distance
                feat = F.normalize(feat, dim=-1).squeeze().cpu().numpy()
            embeddings.append(feat)
            labels.append(label)
        except Exception as e:
            print(f"  ⚠️ Skipping {path}: {e}")

    return np.array(embeddings), np.array(labels)

# 2. Extract all necessary embeddings
# (Assumes train_samples and test_samples exist from your earlier dataset split)
print("Extracting current train embeddings (for KNN index)...")
train_embeddings_final, _ = extract_embeddings_final(train_samples, desc="Train")

print("\nExtracting current test embeddings (Seen classes)...")
test_embeddings_final, _ = extract_embeddings_final(test_samples, desc="Test")

print("\nExtracting unseen embeddings (Out-of-Distribution classes)...")
unseen_embeddings, _ = extract_embeddings_final(unseen_samples, desc="Unseen")

# 3. Balance Test and Unseen sets to equal size for fair F1 calculation
n_eval = min(len(test_embeddings_final), len(unseen_embeddings))

if n_eval == 0:
    raise ValueError("Error: Either Test or Unseen dataset is empty. Cannot perform evaluation.")

np.random.seed(42)
idx_test   = np.random.choice(len(test_embeddings_final), size=n_eval, replace=False)
idx_unseen = np.random.choice(len(unseen_embeddings),      size=n_eval, replace=False)

test_bal   = test_embeddings_final[idx_test]
unseen_bal = unseen_embeddings[idx_unseen]

print(f"\n✅ Balanced Evaluation: {n_eval} Seen vs {n_eval} Unseen")

# 4. Fit KNN and Query Distances
K = 5
knn = NearestNeighbors(n_neighbors=K, metric="cosine", algorithm="brute")
knn.fit(train_embeddings_final)

test_dists, _   = knn.kneighbors(test_bal)
unseen_dists, _ = knn.kneighbors(unseen_bal)

test_mean_dist   = test_dists.mean(axis=1)
unseen_mean_dist = unseen_dists.mean(axis=1)

# 5. Threshold Sweep for Anomaly Detection
all_dists = np.concatenate([test_mean_dist, unseen_mean_dist])
all_true  = np.array([1]*n_eval + [0]*n_eval) # 1=Seen, 0=Unseen

# Generate 300 possible thresholds based on actual distance ranges
thresholds = np.linspace(all_dists.min(), all_dists.max(), 300)
f1_scores, precisions, recalls, unseen_rejected = [], [], [], []

for thresh in thresholds:
    accepted = all_dists < thresh
    tp = ((accepted)  & (all_true == 1)).sum()
    fp = ((accepted)  & (all_true == 0)).sum()
    tn = ((~accepted) & (all_true == 0)).sum()
    fn = ((~accepted) & (all_true == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    f1_scores.append(f1)
    precisions.append(precision)
    recalls.append(recall)
    unseen_rejected.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

best_idx    = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]

print(f"\n🏆 Best OOD Threshold: {best_thresh:.4f}")
print(f"   F1 Score: {f1_scores[best_idx]*100:.1f}%")
print(f"   Unseen Rejected: {unseen_rejected[best_idx]*100:.1f}%")

# 6. Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Metrics vs Threshold
ax = axes[0]
ax.plot(thresholds, np.array(f1_scores)*100, label="F1", color="purple", lw=2.5)
ax.plot(thresholds, np.array(precisions)*100, label="Precision", color="steelblue", lw=1.5, linestyle="--")
ax.plot(thresholds, np.array(recalls)*100, label="Recall (Seen)", color="green", lw=1.5, linestyle="--")
ax.plot(thresholds, np.array(unseen_rejected)*100, label="Unseen Rejected %", color="tomato", lw=1.5, linestyle="--")
ax.axvline(best_thresh, color="black", linestyle=":", label=f"Best Thresh ({best_thresh:.3f})")
ax.set_title(f"Detection Performance vs. Distance\n(Mode: {MODE})")
ax.set_xlabel("Mean Cosine Distance (KNN)")
ax.set_ylabel("Percentage (%)")
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Plot B: Distance Distribution
ax = axes[1]
ax.hist(test_mean_dist, bins=35, alpha=0.6, color="steelblue", label=f"Seen (Test)")
ax.hist(unseen_mean_dist, bins=35, alpha=0.6, color="tomato", label=f"Unseen (OOD)")
ax.axvline(best_thresh, color="black", linestyle=":", label="Decision Boundary")
ax.set_title(f"Distance Distributions: Seen vs Unseen\n(Model: ViT-B/16)")
ax.set_xlabel(f"Mean Cosine Distance to {K} Neighbors")
ax.set_ylabel("Frequency")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(PLOT_DIR, exist_ok=True)
save_path = os.path.join(PLOT_DIR, f"ood_analysis_{MODE}.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Final evaluation complete. Analysis saved to: {save_path}")

In [ ]:
# =============================================================================
# CELL 22 - ROC CURVES + EQUAL ERROR RATE (EER)
# =============================================================================

from sklearn.metrics import roc_curve, auc

# Use the dataset's own mapping — this is the ground truth
idx_to_class = {v: k for k, v in aug_dataset.class_to_idx.items()}

unique_labels = sorted(np.unique(test_labels))
test_labels_bin = label_binarize(test_labels, classes=unique_labels)
test_probs = knn.predict_proba(test_embeddings)

n_classes = len(unique_labels)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

eer_results = {}

for i in range(n_classes):
    fpr, tpr, thresholds = roc_curve(test_labels_bin[:, i], test_probs[:, i])
    roc_auc = auc(fpr, tpr)

    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
    eer_threshold = thresholds[eer_idx]

    # Correct mapping: loop index i → actual label → class name
    actual_label = int(knn.classes_[i])
    class_name = idx_to_class[actual_label]

    eer_results[class_name] = {
        "EER": round(eer, 4),
        "Threshold": round(float(eer_threshold), 4),
        "AUC": round(roc_auc, 4)
    }

    axes[i].plot(fpr, tpr, lw=2, label=f"AUC={roc_auc:.2f}")
    axes[i].scatter(fpr[eer_idx], tpr[eer_idx], color="red", zorder=5, label=f"EER={eer:.2f}")
    axes[i].plot([0, 1], [1, 0], "k--", lw=1, alpha=0.4)
    axes[i].plot([0, 1], [0, 1], "gray", lw=1, linestyle=":", alpha=0.4)
    axes[i].set_title(f"{class_name}\nAUC={roc_auc:.2f}  EER={eer:.2f}", fontsize=9)
    axes[i].set_xlabel("FPR")
    axes[i].set_ylabel("TPR")
    axes[i].set_xlim([0, 1])
    axes[i].set_ylim([0, 1])
    axes[i].legend(fontsize=7)
    axes[i].grid(True)

for j in range(n_classes, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("ROC Curves — BioCLIP + LoRA (One-vs-Rest)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"roc_curve_vit16_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

# Print EER summary table
print(f"\n{'Class':<35} {'EER':>6} {'Threshold':>10} {'AUC':>6}")
print("-" * 60)
for class_name, metrics in eer_results.items():
    print(f"{class_name:<35} {metrics['EER']:>6.4f} {metrics['Threshold']:>10.4f} {metrics['AUC']:>6.4f}")

mean_eer = np.mean([v["EER"] for v in eer_results.values()])
print("-" * 60)
print(f"{'Mean EER':<35} {mean_eer:>6.4f}")
